# **LONG SHORT TERM MEMORY (LSTM)**


## Import Package

In [ ]:
# =====================================================
# LSTM EMPIRIS DENGAN GRID SEARCH
# =====================================================

import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout # type: ignore
from tensorflow.keras.callbacks import EarlyStopping # pyright: ignore[reportMissingModuleSource]
from tensorflow.keras.optimizers import Adam # pyright: ignore[reportMissingModuleSource]
from tensorflow.keras import backend as K # pyright: ignore[reportMissingModuleSource]

## Load Data

In [ ]:
# Jalankan 01_preprocessing.ipynb terlebih dahulu.
import os, pickle, shutil

# Di Kaggle: data berasal dari dataset -> salin ke working directory.
SRC = "/kaggle/input/thesis-indobert-processed-data"
if os.path.exists(SRC):
    shutil.copy(f"{SRC}/split_data.pkl", "split_data.pkl")

with open("split_data.pkl", "rb") as file:
    split_data = pickle.load(file)

X_train_lstm = split_data["X_train_lstm"]
X_test_lstm = split_data["X_test_lstm"]
X_train_bert = split_data["X_train_bert"]
X_test_bert = split_data["X_test_bert"]
y_train = split_data["y_train"]
y_test = split_data["y_test"]


In [ ]:
from pathlib import Path
import pickle   

# =====================================================
# Set seed
# =====================================================
seed = 42
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)

path_split_data = Path("split_data.pkl")

with open(path_split_data, "rb") as f:
    data = pickle.load(f)

# Ekstrak data dari dictionary
df_train = data["df_train"]
df_test = data["df_test"]
X_train_lstm = data["X_train_lstm"]
y_train = data["y_train"]

X_training_lstm = data["X_training_lstm"]
X_validation_lstm = data["X_validation_lstm"]
X_test_lstm = data["X_test_lstm"]

y_training_lstm = data["y_training_lstm"]
y_validation_lstm = data["y_validation_lstm"]
y_test = data["y_test"]

## Tokenizer & padding

In [ ]:

max_words = 10000
max_len = 50

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_training_lstm)

X_train_seq = tokenizer.texts_to_sequences(X_training_lstm)
X_val_seq   = tokenizer.texts_to_sequences(X_validation_lstm)
X_test_seq  = tokenizer.texts_to_sequences(X_test_lstm)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad   = pad_sequences(X_val_seq, maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

## LSTM (tanpa imbalancing data)

Model ini digunakan untuk menguji coba batch_size, drop_out, learning rate dan units yang berbeda

In [ ]:
import random
import itertools
from itertools import product
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense, Dropout, Embedding, LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

# Set seed global
seed = 42

# =====================================================
# Definisi Strategi Imbalanced Data & Hyperparameter
# =====================================================
imbalance_strategies = [
    "Baseline",
    "Class Weight",
    "Random Oversampling",
    "Random Undersampling",
    "SMOTE",
]

param_options = {
    "batch_size": [16, 32],
    "dropout": [0.2, 0.3],
    "learning_rate": [5e-5, 1e-4, 2e-4],
    "units": [32, 64],
}

# Kombinasi hyperparameter (24 kombinasi)
keys = param_options.keys()
param_combinations = list(product(*param_options.values()))
param_grid = [dict(zip(keys, v)) for v in param_combinations]

total_experiments = len(imbalance_strategies) * len(param_grid)
print(f"Total strategi imbalance: {len(imbalance_strategies)}")
print(f"Total kombinasi hyperparameter: {len(param_grid)}")
print(f"Total seluruh eksperimen yang akan diuji: {total_experiments}\n")

# Variabel pelacak performa terbaik
overall_best_f1 = -1
overall_best_model = None
overall_best_params = None
overall_best_strategy = None

# List penampung semua log hasil eksperimen
all_results = []

# =====================================================
# Outer Loop: Iterasi Strategi Imbalanced Data
# =====================================================
for strategy in imbalance_strategies:
    print("\n" + "=" * 60)
    print(f" STRATEGI IMBALANCED DATA: {strategy}")
    print("=" * 60)

    # 1. Penyiapan Data & Parameter Khusus Tiap Strategi (HANYA PADA TRAIN SET)
    X_tr_res, y_tr_res = X_train_pad.copy(), y_training_lstm.copy()
    class_weight_dict = None

    if strategy == "Class Weight":
        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(y_training_lstm),
            y=y_training_lstm,
        )
        class_weight_dict = dict(enumerate(weights))
        print(f"Class Weights Calculated: {class_weight_dict}")

    elif strategy == "Random Oversampling":
        ros = RandomOverSampler(random_state=seed)
        X_tr_res, y_tr_res = ros.fit_resample(X_train_pad, y_training_lstm)

    elif strategy == "Random Undersampling":
        rus = RandomUnderSampler(random_state=seed)
        X_tr_res, y_tr_res = rus.fit_resample(X_train_pad, y_training_lstm)

    elif strategy == "SMOTE":
        # SMOTE akan menghasilkan nilai float pada fitur sintesis integer sequence.
        # Konversi ke int32 agar sesuai dengan layer Embedding Keras.
        smote = SMOTE(random_state=seed)
        X_tr_res, y_tr_res = smote.fit_resample(X_train_pad, y_training_lstm)
        X_tr_res = np.round(X_tr_res).astype("int32")

    print(
        f"Jumlah data training setelah penanganan: {X_tr_res.shape[0]} sampel"
    )

    # =====================================================
    # Inner Loop: Grid Search Hyperparameter
    # =====================================================
    for i, params in enumerate(param_grid):
        exp_number = len(all_results) + 1
        print(
            f"\n--- Eksperimen {exp_number}/{total_experiments} | Strategi: {strategy} | Config #{i+1} ---"
        )
        print(params)

        # Reset session & seed agar deterministik
        K.clear_session()
        tf.random.set_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        # Build Model LSTM
        model = Sequential()
        model.add(
            Embedding(
                input_dim=max_words, output_dim=128, input_length=max_len
            )
        )
        model.add(LSTM(params["units"]))
        model.add(Dropout(params["dropout"]))
        model.add(Dense(64, activation="relu"))
        model.add(Dense(3, activation="softmax"))

        model.compile(
            loss="sparse_categorical_crossentropy",
            optimizer=Adam(learning_rate=params["learning_rate"]),
            metrics=["accuracy"],
        )

        early_stop = EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        )

        # Training Model
        history = model.fit(
            X_tr_res,
            y_tr_res,
            validation_data=(X_val_pad, y_validation_lstm),
            epochs=20,
            batch_size=params["batch_size"],
            class_weight=class_weight_dict,  # None kecuali strategi "Class Weight"
            callbacks=[early_stop],
            verbose=0,  # Ubah ke 1 jika ingin melihat log epoch
        )

        # Predict Validation Set (Test Set tetap murni/tanpa resampling)
        y_val_pred = np.argmax(model.predict(X_val_pad, verbose=0), axis=1)
        _, _, f1_macro, _ = precision_recall_fscore_support(
            y_validation_lstm, y_val_pred, average="macro", zero_division=0
        )

        print(f"Validation Macro F1: {f1_macro:.4f}")

        # Simpan Log Hasil
        all_results.append(
            {
                "strategy": strategy,
                "batch_size": params["batch_size"],
                "dropout": params["dropout"],
                "learning_rate": params["learning_rate"],
                "units": params["units"],
                "val_f1_macro": f1_macro,
            }
        )

        # Evaluasi Kombinasi Terbaik Global
        if f1_macro > overall_best_f1:
            overall_best_f1 = f1_macro
            overall_best_model = model
            overall_best_params = params
            overall_best_strategy = strategy

# =====================================================
# Menampilkan Ringkasan Hasil Eksperimen
# =====================================================
df_results = pd.DataFrame(all_results)

### Accuracy Model

In [ ]:
print("\n" + "=" * 60)
print("HASIL TERBAIK KESELURUHAN")
print("=" * 60)
print(f"Strategi Imbalance Terbaik : {overall_best_strategy}")
print(f"Hyperparameter Terbaik     : {overall_best_params}")
print(f"Best Validation Macro F1   : {overall_best_f1:.4f}")

# Tampilkan 10 Kombinasi Teratas
print("\nTop 10 Kombinasi Eksperimen Terbaik:")
print(df_results.sort_values(by="val_f1_macro", ascending=False).head(10).to_string(index=False))